# Performing Domain Analysis with AI  
## Leveraga SA Model, Write Up, and Results on Strategy to develop a first cut Architecture

First we load up the PEAK Model

### Model-Specific Code (Do Not Modify)

This section contains code that is specific to the system model. It is updated only when the model is changed and should not require user modifications under normal circumstances.

If a new model is introduced, ensure this section is reviewed and updated as needed.


In [1]:
#!pip install --upgrade git+https://github.com/tkSDISW/Capella_Tools 
import capellambse.decl

from capella_tools import capellambse_helper

from IPython import display as diag_display
resources = {
    "PEAK": "PEAK/PEAK",
}
path_to_model = "../PEAK.aird"
model = capellambse.MelodyModel(path_to_model, resources=resources)
from capella_tools import Pub4C
# Instantiate the class with the traceability file
traceability_store = Pub4C.Traceability_Store("../PEAK/traceability")


## 🔄 Embedding Generation Process

Next we load evaluate whether we ne to update the embedings. If there older than model we will update thems. 🚀


In [2]:

from capella_tools  import capella_embeddings_manager

# Generate embeddings for all objects
model_embedding_manager = capella_embeddings_manager.EmbeddingManager()

embedding_file = "embeddings.json" 
model_embedding_manager.set_files( path_to_model , embedding_file)

model_embedding_manager.create_model_embeddings(model)

OpenAI API Key retrieved successfully.
Loading embeddings
embeddings loaded.


## 🎯 Define the model element or diagram for analysis.

We use the embedding to locate a diagram that will be used for basis of analysis. It a OCB diagrams with relations to all the functional chains. 

> 💡 **Tip:** If you're unsure about the model structure, review the documentation or refer to the model's diagrams for additional guidance.


In [3]:

selected_objects = model_embedding_manager.query_and_select_top_objects("[SAB] Water Generation QAS and Params", top_n=1)



This is a list of ranked Objects Based on Query:
Index: 0, Name: [SAB] Water Generation QAS requirements and params, Similarity: 0.88, Type: Diagram, Phase: System Analysis SA, Source: , Target: 


## 📝 Generate Structured Input File 

We then generate the structured input file for all the matching objects and there related objects.
A file "capella_model.yaml" is written it you want to look at it. 

In [4]:
#Workflow
from capella_tools import capellambse_yaml_manager
yaml_handler = capellambse_yaml_manager.CapellaYAMLHandler()
   
#Generate YAML for the logical component and append to the file
for object in  selected_objects :  
    yaml_handler.generate_yaml(model.by_uuid(object["uuid"]))  

yaml_handler.generate_traceability_related_objects(model,traceability_store)

#yaml_handler.display()
yaml_handler.generate_yaml_referenced_objects()
#yaml_handler.display()

yaml_handler.write_output_file()


## ⚙️ Execute  Prompt


Execute the prompt that will use the model and system document. 

In [5]:
import os
from openai import OpenAI
from IPython.core.display import HTML
from IPython.display import display, clear_output, Markdown, IFrame
from capella_tools  import Open_AI_RAG_manager


#print(object)
# Step 1: Get YAML content
yaml_content = yaml_handler.get_yaml_content()

# Step 2: Invoke ChatGPT for analysis
analyzer = Open_AI_RAG_manager.ChatGPTAnalyzer(yaml_content)#analyzer.add_text_file_to_messages("PEAK System.docx")
prompt = """
Write image description for the visually impaired based on the content of this diagram. Specifically call attention to how the PEAK supports Water Management functional chain.
"""
analyzer.follow_up_prompt(prompt)
chatgpt_response = analyzer.get_response()

OpenAI API Key retrieved successfully.


**Your prompt:** 
Write image description for the visually impaired based on the content of this diagram. Specifically call attention to how the PEAK supports Water Management functional chain.


**Response:**

The diagram appears to be a detailed system model that outlines the relationships and interactions among various elements within a water management framework. The primary focus is on the "PEAK" system component and its role in supporting the Water Management functional chain. 

In the diagram, the PEAK system component is positioned as a pivotal element in the water management process. It is directly linked with several system functions such as "Provide Water," which is responsible for distributing water to different recipients within the system, including local populations, aid personnel, and operational user communities. The PEAK supports these functions by providing necessary resources and coordination, ensuring efficient water production, filtering, and distribution.

Several functional exchanges associated with PEAK facilitate the Water Management functional chain. These exchanges are identified by their UUIDs and include interactions like "Manage Water Production," which involves overseeing the processes involved in water filtration and subsequent delivery as "Filtered Water" to the "Receive Water" functions. These exchanges highlight how PEAK-equipped system functions effectively produce and manage the flow of water through various system components.

Additionally, PEAK is involved in other functional chains such as "Provide Situational Awareness" and "Provide Communication Management," indicating its broad capacity in handling multiple aspects of system operations, including situational monitoring, and maintaining communication pathways.

The diagram demonstrates a highly integrated system where PEAK, through its numerous sub-functions and interactions, plays a central role in optimizing water management, making it crucial for system efficiency and resilience in water-related operations.

**Token Usage Info:**

Tokens used: prompt=37960, completion=296, total=38256

## 💬 Launch Interactive Chat on Structured Input




In [6]:
print("Done")

Done


# 